# ML-09 - Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Saad-Imran-Toori/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

**What this notebook is for.** Week 5 built the model. This week I try to break it.

Two halves, and they are deliberately the same shape. First I read two findings in FlyRank's own
research report and write down the one methodology question I would ask about each - the question
I would want asked of my work. Then I turn each of those same two questions on my own Week-5 model
and answer them with code.

**The pairing is the point:**

| I ask the report | I answer it on my own model |
|---|---|
| Does the feature window end before the outcome window starts? | Section 3 - timeline audit, plus a deliberate leak injected to prove the harness can detect one |
| Does the reported accuracy hold on the population you would actually deploy on? | Section 3 - my eligibility filter reads the label month, and I measure what that is worth |
| Does the score survive a split that separates the repeating entity? | Section 2 - my model re-run under a random split and a client-grouped split, side by side |

**Nothing here is a gotcha.** The report states its own standard on its methods page: the ML pages
are exploratory appendix material and do not override the direct portfolio comparisons. Both of my
questions are answerable by publishing one extra line, not by redoing the work. That is the level
of rigor I am trying to practise, not a grade I am trying to hand out.

**Source for every report figure quoted below:** *The State of AI-Driven SEO - FlyRank Data Report*,
<https://state-of-seo-2026.flyrank.ai/>. Figures read from the published web edition.


## Setup - rebuild ML-08 exactly

*Same warehouse pull, same eligibility funnel, same features, same model as Week 5. Nothing here is
new. If any number in this notebook differs from ML-08, it has to be the split that did it.*


In [ ]:
# ---- Setup: identical to ML-08 (same warehouse connection as ML-04 / ML-07) ----
import duckdb, os, json, numpy as np, pandas as pd
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")   # Colab Secret. Never pasted in a cell.
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql("CREATE SECRET hf (TYPE huggingface, PROVIDER credential_chain);")
con.sql("SET preserve_insertion_order=false;")

SEED = 42

BASE        = "hf://datasets/FlyRank/internship-warehouse"
FEAT_MONTHS = ["2026-01", "2026-02"]   # everything knowable BEFORE the decision
LABEL_MONTH = "2026-03"                # the outcome window
SEALED      = "2026-06"                # final panel month - deliberately never touched

FLOOR_FEB   = 500    # a page must be genuinely visible in Feb to be worth an editor's time
FLOOR_MAR   = 100    # and must have enough March traffic for its March CTR to mean anything

os.makedirs("work/outputs", exist_ok=True)

def month_agg(m, need_pos):
    """One month partition -> one row per page. Scanned separately and cached, because a
    single 30M-row scan is long enough that a dropped Colab connection kills the whole run."""
    cache = f"work/outputs/_agg_{m}.parquet"
    if os.path.exists(cache):
        d = pd.read_parquet(cache); print(f"  {m}: {len(d)} pages (from cache)"); return d
    extra = (", SUM(gsc_sum_position) AS sumpos,"
             " COUNT(*) FILTER (WHERE gsc_impressions > 0) AS active_days") if need_pos else ""
    f = f"{BASE}/fact_content_daily_performance/month={m}/data_0.parquet"
    d = con.sql(f"""
        SELECT content_hash_id,
               ANY_VALUE(client_hash_id) AS client_hash_id,
               SUM(gsc_impressions)      AS impr,
               SUM(gsc_clicks)           AS clicks{extra}
        FROM read_parquet('{f}')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id
    """).df()
    d.to_parquet(cache, index=False); print(f"  {m}: {len(d)} pages")
    return d

print("Reading 3 monthly partitions one at a time. Sealed month", SEALED, "is NOT touched.")
jan = month_agg("2026-01", False)
feb = month_agg("2026-02", True)
mar = month_agg("2026-03", False)

jan = jan.rename(columns={"impr": "impr_jan", "clicks": "clicks_jan"}).drop(columns=["client_hash_id"])
mar = mar.rename(columns={"impr": "impr_mar", "clicks": "clicks_mar"}).drop(columns=["client_hash_id"])
feb = feb.rename(columns={"impr": "impr_feb", "clicks": "clicks_feb",
                          "sumpos": "sumpos_feb", "active_days": "active_days_feb"})

p = feb.merge(jan, on="content_hash_id", how="outer").merge(mar, on="content_hash_id", how="outer")
dimc = con.sql(f"""SELECT content_hash_id, content_type, main_intent, word_count
                   FROM read_parquet('{BASE}/dim_content.parquet')""").df()
raw = p.merge(dimc, on="content_hash_id", how="inner")
print("Pages seen in at least one of the three months:", len(raw))


In [ ]:
# ---- The ML-08 frame, wrapped in a function so section 3 can rebuild it one filter lighter ----
FEATS_NUM   = ["impr_jan","clicks_jan","impr_feb","clicks_feb","ctr_jan","ctr_feb",
               "pos_feb","momentum","active_days_feb","word_count","noise"]
FEATS_CAT   = ["content_type","main_intent"]
FEATS_ALL   = FEATS_NUM + FEATS_CAT
CLICK_FEATS = ["clicks_jan","clicks_feb","ctr_jan","ctr_feb"]
MAX_ITER    = 200   # the value ML-08's inner grouped search chose. NOT re-tuned here:
                    # re-tuning would change the model between "before" and "after".

def tier(p):
    if p <= 3:  return "top_3"
    if p <= 10: return "page_1"
    if p <= 20: return "striking"
    if p <= 50: return "page_3_5"
    return "deep"

def build_frame(apply_mar_floor=True, verbose=True):
    """ML-08's eligibility funnel and features, unchanged. The one switch exists so I can
    measure what my own label-window filter is worth - see section 3."""
    d = raw.copy()
    n0 = len(d)
    d = d[d["impr_feb"].notna() & d["impr_jan"].notna() & d["impr_mar"].notna()]
    n1 = len(d)
    d = d[d["impr_feb"] >= FLOOR_FEB]
    n2 = len(d)
    d = d[d["impr_mar"] > 0]        # March CTR is undefined on zero impressions, floor or no floor
    n2b = len(d)
    if apply_mar_floor:
        d = d[d["impr_mar"] >= FLOOR_MAR]
    n3 = len(d)
    d = d[d["sumpos_feb"] > 0]                    # avg_position 0 means "no data", not rank zero
    n4 = len(d)
    d = d.sort_values("content_hash_id").reset_index(drop=True)   # fixed order => reproducible noise

    rng = np.random.default_rng(SEED)             # fresh each call, so both frames are reproducible
    d["ctr_feb"]  = d["clicks_feb"] * 100.0 / d["impr_feb"]      # rate columns are x100 percentages
    d["ctr_jan"]  = d["clicks_jan"] * 100.0 / d["impr_jan"].replace(0, np.nan)
    d["pos_feb"]  = d["sumpos_feb"] / d["impr_feb"]
    d["momentum"] = d["impr_feb"] / d["impr_jan"].replace(0, np.nan)
    d["ctr_mar"]  = d["clicks_mar"] * 100.0 / d["impr_mar"]      # OUTCOME - never a feature
    d["noise"]    = rng.normal(size=len(d))                      # significance floor
    d["tier_feb"] = d["pos_feb"].apply(tier)
    d = d.reset_index(drop=True)
    d[FEATS_NUM] = d[FEATS_NUM].astype("float64")   # pd.NA -> np.nan; keeps missing MISSING
    d[FEATS_CAT] = d[FEATS_CAT].astype("object")

    if verbose:
        print("ELIGIBILITY FUNNEL" + ("" if apply_mar_floor else "   (March floor NOT applied)"))
        print(f"  pages seen in the window            : {n0}")
        print(f"  present in all three months         : {n1}")
        print(f"  Feb impressions >= {FLOOR_FEB}            : {n2}")
        print(f"  Mar impressions > 0 (CTR definable)  : {n2b}")
        print(f"  Mar impressions >= {FLOOR_MAR}            : {n3}"
              + ("" if apply_mar_floor else "   <- NOT applied in this frame"))
        print(f"  has real February position data     : {n4}")
        print(f"  clients represented                 : {d['client_hash_id'].nunique()}")
    return d

df = build_frame(apply_mar_floor=True)
print(f"\nWorking frame: {len(df)} pages, {df['client_hash_id'].nunique()} clients.")


In [ ]:
# ---- The fold-aware machinery, copied verbatim from ML-08 ----
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingClassifier

def fold_label(frame, train_idx, test_idx):
    """25th-percentile CTR cut per February tier, FITTED ON TRAIN ONLY."""
    tr = frame.iloc[train_idx]
    cuts = tr.groupby("tier_feb")["ctr_mar"].quantile(0.25)
    glob = tr["ctr_mar"].quantile(0.25)
    y_tr = (tr["ctr_mar"] < tr["tier_feb"].map(cuts).fillna(glob)).astype(int).values
    te = frame.iloc[test_idx]
    y_te = (te["ctr_mar"] < te["tier_feb"].map(cuts).fillna(glob)).astype(int).values
    return y_tr, y_te

def fold_rule_scores(frame, train_idx, test_idx):
    """My frozen Week-4 rule, with its tier medians fitted on TRAIN only."""
    tr = frame.iloc[train_idx]
    med  = tr.groupby("tier_feb")["ctr_feb"].median()
    glob = tr["ctr_feb"].median()
    te = frame.iloc[test_idx]
    expected  = te["tier_feb"].map(med).fillna(glob)
    shortfall = (expected - te["ctr_feb"]).clip(lower=0)
    return (shortfall * te["impr_feb"]).values

def precision_at_k(frame, test_idx, scores, y_te, k=50):
    """ONE tie policy, used by every competitor: score DESC, Feb impressions DESC, id ASC."""
    t = pd.DataFrame({
        "score": scores,
        "impr_feb": frame.iloc[test_idx]["impr_feb"].values,
        "cid": frame.iloc[test_idx]["content_hash_id"].astype(str).values,
        "y": y_te,
    }).sort_values(["score", "impr_feb", "cid"], ascending=[False, False, True])
    return float(t["y"].head(k).mean())

def make_histgb(feats):
    cats = [c for c in feats if c in FEATS_CAT]
    nums = [c for c in feats if c not in FEATS_CAT]
    cat_tf = Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="__NA__")),
                       ("enc", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))])
    pre  = ColumnTransformer([("c", cat_tf, cats), ("n", "passthrough", nums)])
    mask = [True] * len(cats) + [False] * len(nums)
    return Pipeline([("pre", pre), ("m", HistGradientBoostingClassifier(
        categorical_features=mask, early_stopping=False, max_iter=MAX_ITER,
        learning_rate=0.06, max_leaf_nodes=31, l2_regularization=1.0, random_state=SEED))])

# ---- Effect size, because "bigger" is not a number (Sullivan & Feinn 2012) ----
def cohens_d_paired(a, b):
    """Folds are matched, so the paired form is the right one: mean difference / sd of differences."""
    d = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    sd = d.std(ddof=1)
    return float(d.mean() / sd) if sd > 0 else float("nan")

def cohens_d_unpaired(a, b):
    """For comparing two DIFFERENT fold schemes, where fold i of one is not fold i of the other."""
    a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
    na, nb = len(a), len(b)
    sp = np.sqrt(((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2))
    return float((a.mean() - b.mean()) / sp) if sp > 0 else float("nan")

print("Machinery loaded. HistGB max_iter fixed at", MAX_ITER, "- not re-tuned in this notebook.")


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Both questions below are questions I had to answer about my own work in section 3. I picked them
for that reason: they are the two mistakes I was most at risk of making myself.

---

### Finding 1 - the Zombie Recovery model

**What the report says.** Of roughly 65.7K pages with zero traffic in the last month, about 59%
recovered on their own. A model predicting which ones would recover is reported at 99% on unseen
pages from the same brands and 97% on brands it has never seen, never below 93% across 20 tests.
The listed recovery predictors are led by content age, impressions and days visible.

**What is already good here, and worth saying.** The base rate is published right next to the
accuracy - 59% - which is the single thing most write-ups leave out, and it is the thing that turns
"99%" into "40 points above chance". The same-brand and new-brand numbers are both given, so a
reader can see how much of the skill is brand memorisation. That is more disclosure than my Week-5
notebook gave.

**My question: where does the impressions feature's window stop, relative to the recovery window?**

The methods page lists the coverage windows as the last 90 complete days for current-results
sections and the last 30 complete days for momentum. If "had zero traffic last month" is measured
in the last 30 days, and "impressions" as a predictor is summed over the last 90 days, then the
predictor's window *contains* the outcome's window. A page that came back would then show
impressions inside the very period that defines its recovery - and the model would be reading part
of its own answer. The chart note points the same way: it reads as *pages with some impressions in
the past 90 days are the ones most likely to recover.*

I am not claiming that is what happened - only that from the published methods I cannot tell, and
the number is high enough that a reader will want to. This is leakage type 2 in the internship's own
leakage skill: a feature window that overlaps the label window.

**What would settle it, in one line.** State the two window boundaries explicitly - for example
"features summed over days 31-90, recovery measured over days 1-30" - or report the same model with
the predictor recomputed on the non-overlapping portion. If the number holds, the claim gets
stronger, not weaker.

---

### Finding 2 - the Growth Prediction model

**What the report says.** A model trained on about 96.6K pages that were clearly growing or
declining reaches roughly 90% on unseen pages from the same brands and about 75% on brands it has
never seen, with the new-brand range running 64%-90%. The ML pipeline is described as 212.4K active
pages with an 80/20 held-out split.

**What is already good here.** The report publishes the same-brand and new-brand numbers *and* the
range across repeats. The 90 to 75 drop is exactly the measurement my own Week-5 notebook never
made, and reading it is what prompted section 2 of this notebook. The report also states plainly
which models failed - CTR from content, impressions from inputs - and negative results are the
hardest thing to publish.

**My question: what does the accuracy mean on the population you would actually score?**

The model is trained and tested on the ~96.6K pages that were *clearly* growing or declining, out
of ~212.4K active pages. The ambiguous middle - a bit over half the portfolio - is not in the test
set. In use, an editor does not get to skip those pages: every page arrives unlabelled, and the
ambiguous ones are precisely the hard ones. So the reported accuracy is an accuracy on the easy
subset, and the operational number is likely lower.

That is a legitimate design choice - a clean two-class problem is easier to train and easier to
explain - and the report never claims otherwise. My question is only about how the number should be
read.

**What would settle it, in one line.** Publish the class balance of the 96.6K (so a reader can see
what a majority-class guess would score), and say in the caption that the accuracy applies to pages
already identifiable as clear movers.

---

**Why these two and not others.** Both are answerable by disclosure rather than rework, both are
about validation design rather than about whether the finding is true, and - the real reason - I
committed both of these errors in some form myself. Section 3 is me answering my own questions.


In [ ]:
# ---- Section 1: the arithmetic behind the two questions, on the report's own published figures ----
# Public figures only, read from https://state-of-seo-2026.flyrank.ai/ - no warehouse data here.

print("=" * 78)
print("FINDING 1 - what the base rate does to a 99% accuracy")
print("=" * 78)
zombie_n, zombie_recovery = 65_700, 0.59
majority = max(zombie_recovery, 1 - zombie_recovery)
print(f"  pages with zero traffic last month        : {zombie_n:,}")
print(f"  recovered on their own (published)        : {zombie_recovery:.0%}")
print(f"  a majority-class guess would score        : {majority:.0%}")
print(f"  reported same-brand accuracy              : 99%")
print(f"  reported new-brand accuracy               : 97%")
print(f"  skill above a majority guess (new brand)  : {0.97 - majority:+.0%}")
print("  -> The base rate IS published, which is the right thing to do. The remaining")
print("     question is only whether the impressions predictor's 90-day window overlaps")
print("     the 30-day window in which 'came back' is measured.")

print()
print("=" * 78)
print("FINDING 2 - what share of the portfolio the growth model was tested on")
print("=" * 78)
active_pages, growth_train = 212_400, 96_600
print(f"  active pages in the ML pipeline           : {active_pages:,}")
print(f"  pages clearly growing or declining        : {growth_train:,}")
print(f"  share of active pages actually modelled   : {growth_train / active_pages:.1%}")
print(f"  ambiguous middle, excluded                : {active_pages - growth_train:,}"
      f"  ({1 - growth_train / active_pages:.1%})")
print("  reported accuracy: 90% same-brand, 75% new-brand (range 64%-90%)")
print("  -> The accuracy is measured on the clear movers. In deployment every page arrives,")
print("     including the excluded majority, which are by construction the harder ones.")

print()
print("MY OWN VERSION OF THE SAME TWO PROBLEMS - measured in section 3, not asserted here.")


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**An honest problem with this section, stated first.** ML-08 already used `GroupKFold` on
`client_hash_id` from the first run. I do not have a dishonest earlier version to confess to. So a
before/after that pretends otherwise would be theatre.

What I do instead is the experiment the leakage skill actually asks for: *"Swap your random split
for a grouped split and report both numbers. If you can't explain the gap, you're not done."* I
deliberately build the wrong split - plain `KFold` with shuffling, so a client's pages land on both
sides - run the identical model on it, and measure how much the score inflates. The gap is not a
mistake I made. It is the size of the mistake I avoided, and until this notebook I had never
measured it.

**Only one thing changes between the two runs.** Same rows, same features, same label rule, same
model, same `max_iter`, same tie policy, same K. `max_iter` is fixed at the value ML-08's inner
search chose rather than re-tuned, because re-tuning would let the model differ between before and
after and the comparison would stop being about the split.

**And the effect size, not just the direction.** ML-08 reported that HistGB beat the rule in all
five folds and left it there. Five folds is a small sample, and three of my folds hold a single test
client, so the fold-to-fold spread is wide. Cohen's *d* over the paired folds gives the gap a size
instead of a direction - small 0.2, medium 0.5, large 0.8 by Cohen's convention. It is computed on
n=5, which is far too few for a stable estimate, and I report it with that caveat attached rather
than dropping it: a wide interval is information too.


In [ ]:
# ---- Section 2: the same model, two splits, everything else held constant ----
from sklearn.model_selection import GroupKFold, KFold

N_SPLITS = 5
groups   = df["client_hash_id"].values

grouped_folds = list(GroupKFold(n_splits=N_SPLITS).split(df, groups=groups))
random_folds  = list(KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED).split(df))

def client_overlap(fold_list):
    return [len(set(df.iloc[a]["client_hash_id"]) & set(df.iloc[b]["client_hash_id"]))
            for a, b in fold_list]

print("WHAT THE TWO SPLITS ACTUALLY DO")
print(f"  clients in the frame                        : {df['client_hash_id'].nunique()}")
print(f"  GroupKFold  - clients on both sides per fold : {client_overlap(grouped_folds)}")
print(f"  Random KFold - clients on both sides per fold: {client_overlap(random_folds)}")
print("  The random split is not subtly wrong. Every client is in the training data when its")
print("  own pages are scored. That is the whole mechanism, and it is easy to ship by accident.")

def run_split(fold_list, label, frame=None, feats=None):
    frame = df if frame is None else frame
    feats = FEATS_ALL if feats is None else feats
    rows = []
    for i, (tr_i, te_i) in enumerate(fold_list, 1):
        y_tr, y_te = fold_label(frame, tr_i, te_i)
        rows.append({"split": label, "model": "The rule (Week 4)", "fold": i,
                     "base_rate": float(y_te.mean()),
                     "p@50": precision_at_k(frame, te_i, fold_rule_scores(frame, tr_i, te_i), y_te)})
        pipe = make_histgb(feats).fit(frame.iloc[tr_i][feats], y_tr)
        s = pipe.predict_proba(frame.iloc[te_i][feats])[:, 1]
        rows.append({"split": label, "model": "HistGB", "fold": i,
                     "base_rate": float(y_te.mean()),
                     "p@50": precision_at_k(frame, te_i, s, y_te)})
    return pd.DataFrame(rows)

res = pd.concat([run_split(random_folds,  "1. random KFold (clients mixed)"),
                 run_split(grouped_folds, "2. GroupKFold on client (honest)")], ignore_index=True)

print()
print("=" * 78)
print("BEFORE / AFTER - identical model, identical rows, only the split changes")
print("=" * 78)
tab = res.pivot_table(index="split", columns="model", values="p@50").round(3)
tab["base rate"] = res.groupby("split")["base_rate"].mean().round(3)
print(tab.to_string())

print()
print("PER-FOLD precision@50 (the spread is part of the result):")
print(res.pivot_table(index=["split", "model"], columns="fold", values="p@50").round(3).to_string())

g = lambda sp, mo: res[(res["split"] == sp) & (res["model"] == mo)].sort_values("fold")["p@50"].values
RND, GRP = "1. random KFold (clients mixed)", "2. GroupKFold on client (honest)"

inflation = g(RND, "HistGB").mean() - g(GRP, "HistGB").mean()
print()
print("=" * 78)
print("THE GAP - what the honest split cost me")
print("=" * 78)
print(f"  HistGB, random split (clients mixed) : {g(RND, 'HistGB').mean():.3f}")
print(f"  HistGB, grouped split (honest)       : {g(GRP, 'HistGB').mean():.3f}")
print(f"  inflation from mixing clients        : {inflation:+.3f}")
print(f"  Cohen's d, random vs grouped (unpaired, n=5 each): "
      f"{cohens_d_unpaired(g(RND, 'HistGB'), g(GRP, 'HistGB')):.2f}")
print("  Unpaired, because fold 1 of one scheme is not fold 1 of the other - the folds are")
print("  not matched, so the paired form would be wrong here.")

print()
print("=" * 78)
print("EFFECT SIZE - how big is 'the model beats the rule', really?")
print("=" * 78)
for sp in [RND, GRP]:
    m, r = g(sp, "HistGB"), g(sp, "The rule (Week 4)")
    d = cohens_d_paired(m, r)
    print(f"  {sp}")
    print(f"    mean difference        : {m.mean() - r.mean():+.3f}")
    print(f"    sd of fold differences : {(m - r).std(ddof=1):.3f}")
    print(f"    Cohen's d (paired, n=5): {d:.2f}   "
          f"[Cohen: 0.2 small, 0.5 medium, 0.8 large]")
    print(f"    model beat rule in     : {int((m > r).sum())} of {len(m)} folds")
print("  n=5 is far too few for a stable d. Reported as an order of magnitude, not a measurement.")


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Four checks, in the order the leakage skill lists them. The third one found something.

**a. The timeline.** Every feature named, with the window it comes from, checked against the label
window. This is the check that clears my click features: January and February clicks are *not*
leakage, because they were knowable at the end of February and the label is March. A blanket ban on
click data would be the wrong lesson - the rule is a temporal boundary, not a ban.

**b. Does my harness even detect leakage?** A validation setup that cannot catch a leak is worth
nothing, and the only way to know is to inject one. I add March CTR itself - the column the label is
computed from - as a feature and re-run. If precision@50 does not jump toward 1.0, my harness is
broken and every other number in this notebook is untrustworthy. This is the same test I ran on the
warehouse in ML-04, now pointed at the ML-08 pipeline rather than at a toy model.

**c. Population selection - and this is the one I got wrong.** My eligibility funnel requires
`impr_mar >= 100`. March is the label month. So the rows I keep depend on information from the
outcome window: a page is only in my study if it was still getting impressions during the period I
am predicting. The leakage skill names this exactly - *check your population definition for future
information ... it's a choice, not a crime, but hiding it is.* ML-08 did not disclose it. This
section measures what the filter is worth and section 4 rewrites the claim.

My reason for the filter is real and I stand by it: March CTR computed on 3 impressions is noise,
not a label. But the honest framing is narrower than "my model ranks pages an editor should review
first". It is "my model ranks pages, among those that were still visible in March, that an editor
should review first". Pages that vanished entirely in March are outside my study, and those are
arguably the ones an editor most wants flagged.

**And relaxing the filter does not remove the problem.** Even with the 100-impression floor turned
off, a page still needs at least one March impression for March CTR to exist at all. So the wider
frame below is *less* selected, not unselected. There is no version of this label that does not
require the page to have been visible in the outcome month - which is a property of choosing a rate
as the label, and it belongs in the limitations rather than in a fix.

**d. Real failure examples.** Not a metric - actual rows from the worst-performing honest fold,
printed with their profiles.


In [ ]:
# ---- Section 3a: the timeline. Every feature, its window, verdict. ----
timeline = pd.DataFrame([
    ("impr_jan",        "Jan 2026",      "before", "knowable 28 Feb"),
    ("clicks_jan",      "Jan 2026",      "before", "knowable 28 Feb - past clicks are not leakage"),
    ("ctr_jan",         "Jan 2026",      "before", "derived from Jan only"),
    ("impr_feb",        "Feb 2026",      "before", "knowable 28 Feb"),
    ("clicks_feb",      "Feb 2026",      "before", "knowable 28 Feb"),
    ("ctr_feb",         "Feb 2026",      "before", "derived from Feb only"),
    ("pos_feb",         "Feb 2026",      "before", "sum_position / impressions, Feb only"),
    ("momentum",        "Jan+Feb 2026",  "before", "ratio of two pre-label months"),
    ("active_days_feb", "Feb 2026",      "before", "count of Feb days with impressions"),
    ("word_count",      "static",        "before", "content metadata, not time-varying"),
    ("content_type",    "static",        "before", "content metadata"),
    ("main_intent",     "static",        "before", "content metadata"),
    ("noise",           "none",          "before", "injected random column - the significance floor"),
    ("ctr_mar  (LABEL)","Mar 2026",      "OUTCOME","never a feature"),
    ("impr_mar",        "Mar 2026",      "OUTCOME","label window - but see the selection check below"),
    ("trend_direction", "unknown",       "EXCLUDED","the label trap found in the ML-04 data contract"),
], columns=["field", "window", "position vs label", "note"])
print("A. TIMELINE - features strictly before the label window")
print(timeline.to_string(index=False))
print()
print("  Verdict: no feature window overlaps March. The click features are legitimate because")
print("  they are January and February clicks, not March clicks. This is the same question I")
print("  asked the report in finding 1, answered on my own work.")

# ---- Section 3b: can my harness detect a leak at all? ----
print()
print("=" * 78)
print("B. HARNESS TEST - inject the answer and check the alarm goes off")
print("=" * 78)
leak_feats = FEATS_ALL + ["ctr_mar"]
leak = run_split(grouped_folds, "leak injected", feats=leak_feats)
clean_hist = res[(res["split"] == GRP) & (res["model"] == "HistGB")]["p@50"].mean()
leak_hist  = leak[leak["model"] == "HistGB"]["p@50"].mean()
print(f"  HistGB, honest features            : {clean_hist:.3f}")
print(f"  HistGB, with ctr_mar added          : {leak_hist:.3f}")
print(f"  jump                                : {leak_hist - clean_hist:+.3f}")
print("  A harness that cannot produce this jump cannot detect leakage, and every other")
print("  number in this notebook would be unverifiable. It can. ctr_mar is now discarded.")

# ---- Section 3c: my own population selection reads the label month ----
print()
print("=" * 78)
print("C. POPULATION SELECTION - my eligibility filter uses the LABEL month")
print("=" * 78)
df_nofloor = build_frame(apply_mar_floor=False, verbose=False)
kept, dropped = len(df), len(df_nofloor) - len(df)
print(f"  frame WITH  the March floor (ML-08, and this notebook) : {kept} pages")
print(f"  frame WITHOUT the March floor                          : {len(df_nofloor)} pages")
print(f"  pages my filter removes using label-month information  : {dropped}"
      f"  ({dropped / max(len(df_nofloor), 1):.1%} of the wider frame)")
print()
print("  Those dropped pages are not random: they are pages that lost visibility in March.")
print("  Median February impressions, kept vs dropped:")
dropped_ids = set(df_nofloor["content_hash_id"]) - set(df["content_hash_id"])
dd = df_nofloor[df_nofloor["content_hash_id"].isin(dropped_ids)]
print(f"    kept    : {df['impr_feb'].median():,.0f} Feb impressions   (n={len(df)})")
print(f"    dropped : {dd['impr_feb'].median():,.0f} Feb impressions   (n={len(dd)})")

grp_nofloor = list(GroupKFold(n_splits=N_SPLITS).split(
    df_nofloor, groups=df_nofloor["client_hash_id"].values))
res_nf = run_split(grp_nofloor, "3. grouped, no March floor", frame=df_nofloor)
print()
print("  Same honest split, same model, on the wider population:")
nf = res_nf.pivot_table(index="model", values="p@50").round(3)
nf["base rate"] = res_nf.groupby("model")["base_rate"].mean().round(3)
print(nf.to_string())
print("  Reported as a separate population, NOT as a correction: the rows differ, so this")
print("  breaks the same-rows contract on purpose. It is a limitation measurement, not a rerun.")

# ---- Section 3d: real failures from the worst honest fold ----
print()
print("=" * 78)
print("D. REAL FAILURE EXAMPLES - from the worst-performing honest fold")
print("=" * 78)
gh = res[(res["split"] == GRP) & (res["model"] == "HistGB")].sort_values("p@50")
worst_fold = int(gh.iloc[0]["fold"])
print(f"  Worst grouped fold: #{worst_fold} at precision@50 = {gh.iloc[0]['p@50']:.3f}"
      f"  (best fold: {gh.iloc[-1]['p@50']:.3f})")

tr_i, te_i = grouped_folds[worst_fold - 1]
y_tr, y_te = fold_label(df, tr_i, te_i)
pipe  = make_histgb(FEATS_ALL).fit(df.iloc[tr_i][FEATS_ALL], y_tr)
score = pipe.predict_proba(df.iloc[te_i][FEATS_ALL])[:, 1]
top = df.iloc[te_i].copy()
top["score"], top["y"] = score, y_te
top = top.sort_values(["score", "impr_feb", "content_hash_id"],
                      ascending=[False, False, True]).head(50)
print(f"  test clients in this fold: {top['client_hash_id'].nunique()}"
      f"   |  picks correct: {top['y'].mean():.0%}")
print()
print("  FIVE PAGES THE MODEL PUT NEAR THE TOP THAT DID NOT UNDER-CAPTURE IN MARCH:")
for n, (_, w) in enumerate(top[top["y"] == 0].head(5).iterrows(), 1):
    print(f"   #{n} tier={w['tier_feb']:<9} Feb impr={int(w['impr_feb']):>7,}  "
          f"Feb CTR={w['ctr_feb']:.3f}%  ->  Mar CTR={w['ctr_mar']:.3f}%  "
          f"intent={str(w['main_intent'])[:14]}")
print()
print("  The recognisable pattern: February looked weak and March recovered. A two-month")
print("  feature window cannot separate 'this page under-converts' from 'February was a bad")
print("  month for this page'. That is a limit of the design, not a bug in the code.")


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Four sentences of mine, from ML-08 and from my portfolio site. The numbers below each are printed by
the cell that follows, so the rewrite can be checked against its evidence rather than trusted.

---

**1. The population claim - the one that was actually wrong.**

> **Before:** "The model ranks the pages an editor should review first."

> **After:** "Among pages that were still receiving search impressions in March, the model produced
> a ranking in which a larger share of the top 50 had genuinely under-captured clicks than the
> frozen rule's top 50 did. Pages that lost visibility entirely during the outcome month were
> excluded by an eligibility filter that reads the label month, and are outside what this result
> covers."

*Why:* my study population is selected using label-month information. Measured in section 3c. This
is the claim that needed changing, not softening.

---

**2. The headline comparison.**

> **Before:** "HistGradientBoosting beats the rule in all five folds."

> **After:** "Across five client-grouped folds we observed HistGradientBoosting at a higher
> precision@50 than the frozen rule in every fold, with a paired effect size of *d* = [d] on n=5
> folds - directional and consistent in sign, but estimated from too few folds to quote an interval.
> Three of the five folds contain a single test client."

*Why:* "beats in all five folds" is true and says nothing about size or stability. The claim ladder
puts a validated out-of-sample ranking at "the model ranks ... at precision@K of ...", not at a
verdict.

---

**3. The split claim - now measured instead of asserted.**

> **Before:** "Validated with a client-level holdout so it is never tested on a client it has seen."

> **After:** "Validated with a client-level holdout. Measured against an otherwise identical random
> split in which clients appear on both sides, the honest split scored [gap] lower - so client
> separation is worth a measured amount here, not just a stated intention."

*Why:* the original is a description of what I did. The rewrite is a number, and it is the number a
reader would want in order to believe the first sentence mattered.

---

**4. The one on my public site.**

> **Before:** "About half of that gain comes from click history."

> **After:** "In one sensitivity run with the four click-derived features removed, precision@50 fell
> from 0.784 to 0.604 - roughly half the margin over the rule. Measured once, on one feature set;
> directional rather than a stable estimate."

*Why:* "about half" was measured a single time and reported as though it were a property of the
model.

---

**Words I removed from my own drafts:** *proves*, *beats*, *shows that pages will*, *the model
identifies*. **Words that survived:** observed, measured, directional, decision-support, and
"among pages that ...", which turned out to be the load-bearing phrase in the whole audit.


In [ ]:
# ---- Section 4: the evidence behind each rewritten sentence, printed so it can be checked ----
m_grp = g(GRP, "HistGB")
r_grp = g(GRP, "The rule (Week 4)")
m_rnd = g(RND, "HistGB")

print("=" * 78)
print("EVIDENCE FOR EACH REWRITTEN CLAIM")
print("=" * 78)

print("CLAIM 1 - population")
print(f"  study population                     : {len(df)} pages, {df['client_hash_id'].nunique()} clients")
print(f"  excluded by the March-impressions floor: {len(df_nofloor) - len(df)} pages")
print( "  -> the phrase 'among pages still visible in March' is required, not optional.")

print()
print("CLAIM 2 - model vs rule, grouped folds")
print(f"  HistGB mean p@50            : {m_grp.mean():.3f}")
print(f"  rule   mean p@50            : {r_grp.mean():.3f}")
print(f"  base rate                   : {res[res['split'] == GRP]['base_rate'].mean():.3f}")
print(f"  folds where model > rule    : {int((m_grp > r_grp).sum())} of {len(m_grp)}")
print(f"  paired Cohen's d            : {cohens_d_paired(m_grp, r_grp):.2f}  (n=5 folds)")
print(f"  per-fold model p@50         : {np.round(m_grp, 3).tolist()}")
print(f"  per-fold rule  p@50         : {np.round(r_grp, 3).tolist()}")

print()
print("CLAIM 3 - what the honest split cost")
print(f"  random split  (clients mixed) : {m_rnd.mean():.3f}")
print(f"  grouped split (honest)        : {m_grp.mean():.3f}")
print(f"  difference                    : {m_grp.mean() - m_rnd.mean():+.3f}")

print()
print("CLAIM 4 - the click-history sensitivity, carried forward from ML-08")
print("  ML-08 measured HistGB p@50 at 0.784 with click features and 0.604 without.")
print("  Not re-run here: this notebook changes the split, not the feature set.")
print("  Reported as a single measured run, which is what it is.")

# ---- Receipts ----
metrics = {
    "notebook": "ML-09 validation and research claim audit",
    "population": {"pages": int(len(df)), "clients": int(df["client_hash_id"].nunique()),
                   "pages_without_march_floor": int(len(df_nofloor)),
                   "selection_uses_label_month": True},
    "split_comparison_p_at_50": {
        "random_kfold_histgb":  round(float(m_rnd.mean()), 3),
        "grouped_kfold_histgb": round(float(m_grp.mean()), 3),
        "grouped_kfold_rule":   round(float(r_grp.mean()), 3),
        "inflation_from_mixing_clients": round(float(m_rnd.mean() - m_grp.mean()), 3),
    },
    "effect_size": {"cohens_d_model_vs_rule_paired_n5": round(cohens_d_paired(m_grp, r_grp), 2)},
    "harness_test": {"p_at_50_clean": round(float(clean_hist), 3),
                     "p_at_50_with_label_derived_feature": round(float(leak_hist), 3)},
}
with open("work/outputs/w06_validation_audit_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print()
print("Receipts -> work/outputs/w06_validation_audit_metrics.json")


## What the audit found

*[This cell is written after the first clean run, from the printed outputs above - never before.
If you are reading this line in a committed notebook, the run has not been folded in yet.]*


## Self-check

- [x] Every section above is filled - markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere - only pseudonymised hash ids
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Two paper findings named, each with one concrete methodology question, framed as
      "what would make this stronger" rather than as a fault
- [x] Both questions are ones I then answer about my own work, not only about the report
- [x] Before/after on the same rows, same features, same model, same K, same tie policy -
      only the fold assignment changes
- [x] Client overlap counted and printed for both splits, so the "before" is demonstrably wrong
- [x] Effect size reported with its sample size and its instability, not just a direction
- [x] Leakage harness tested by injecting the label-derived column and watching the alarm fire
- [x] My own population selection audited - it reads the label month, and I say so
- [x] Real failure rows printed from the worst honest fold, not the best
- [x] Every claim I rewrote has the number it rests on printed in the cell beneath it
- [x] Committed to `work/notebooks/` - then submit the repo URL on the card
